<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/weightspace_ridge_1d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Weight-space ridges vs point inducing (MLP)

CIP-0028 via the `gpc` façade (+ CIP-0031 predict contract). Toy 1-D regression with DTCVAR: point inducing in input space versus ridge windows in whitened weight space.

The finite-\(M\) **weight-space** posterior mean is a bank of smoothed erf units (a finite-width counterpart of the Williams MLP). Ridge centres are free by default (`inducing_fixed=False`, matching C++). Softness defaults to a **tied** scalar `window_width` (fixed unless `learn_window_width=True`). [CIP-0032](../../cip/cip0032.md) adds per-unit widths `s_m` via a length-\(M\) `window_width` array or `learn_window_widths=True` — that restores MLP-like per-unit sharpness, at the cost of a harder \(K_{uu}\) conditioning problem.

`plot_1d` defaults to `predict` (\(y_*\)); pass `noiseless=True` for `predict_noiseless` (\(f_*\)).

```bash
pip install matplotlib   # or: pip install 'gpc[plot]'
```


In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a10"


In [ ]:
%pip install matplotlib


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

rng = np.random.default_rng(0)
X = np.linspace(-1.5, 1.5, 80)[:, None]
Y = np.sin(2.0 * X) + 0.1 * np.cos(3.0 * X) + 0.05 * rng.normal(size=X.shape)
Z = np.linspace(-1.2, 1.2, 12)[:, None]

# Centres free by default. Keep s fixed here; try learn_window_width=True to free softness.
m_pts = gpc.SparseGPRegression(X, Y, Z, kernel="mlp", inducing="points")
m_ws = gpc.SparseGPRegression(
    X, Y, Z, kernel="mlp", inducing="weightspace", window_width=0.55
)
for m in (m_pts, m_ws):
    m.optimize(iters=120)

print("points objective:", m_pts.objective(), "inducing:", m_pts.inducing_type)
print("weightspace objective:", m_ws.objective(), "s=", m_ws.window_width)
print("points fixed?", m_pts.inducing_fixed, "paths:", m_pts.param_paths())
print("weightspace fixed?", m_ws.inducing_fixed, "paths:", m_ws.param_paths())


In [ ]:
# Optional: free global softness s (learn_window_width=True).
# str(m) / param_paths now expose inducing.window_width (+ve when learnt, fixed otherwise).
m_learn = gpc.SparseGPRegression(
    X, Y, Z, kernel="mlp", inducing="weightspace",
    window_width=0.55, learn_window_width=True,
)
print("before:\n", m_learn)
s0 = m_learn.window_width
m_learn.optimize(iters=120)
print("after s:", s0, "→", m_learn.window_width, "obj:", m_learn.objective())
print(m_learn)

In [ ]:
# CIP-0032: per-unit softness s_m (learn_window_widths=True).
# Paths become inducing.window_width.0 .. M-1; str(m) lists them when M is small.
m_untied = gpc.SparseGPRegression(
    X, Y, Z, kernel="mlp", inducing="weightspace",
    window_width=0.55, learn_window_widths=True,
)
print("paths:", [p for p in m_untied.param_paths() if "window_width" in p])
print("before s_m:", m_untied.window_widths)
m_untied.optimize(iters=120)
print("after s_m:", m_untied.window_widths, "obj:", m_untied.objective())
print(m_untied)

In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

Xtest = np.linspace(-1.8, 1.8, 300)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m_pts, X, Y, Xtest=Xtest, ax=axes[0], label="points")
axes[0].set_title("MLP + point inducing")
plot_1d(m_ws, X, Y, Xtest=Xtest, ax=axes[1], label="weightspace")
axes[1].set_title("MLP + weight-space ridges")
fig.suptitle("DTCVAR — same M; free inducing centres (s fixed for ridges)")
plt.tight_layout()
plt.show()
